# 10 Graph v2, stage 1: pilot, then GraphSAGE vs no-edge MLP vs shuffled-edge GraphSAGE on repeat 0 (5 folds)
Pre-registered in `docs/PROJECT_LOG.md` 6f; B0 results in `results/graph_v2_b0.json` (K = 5, email edges kept). Node features = the 79 W2+E+H columns; same recipe as step 5 (2 layers, hidden 64, dropout 0.3, Adam lr 0.01, wd 5e-4, sqrt class weight, last-epoch model, ONE seed (42) per fold). Epoch counts come from one pilot on repeat-0 fold-0 training users (3 inner splits, <= 300 epochs). **Gate:** GraphSAGE beats BOTH the MLP and the shuffled-edge control by a mean PR-AUC margin >= +0.05, positive in >= 4 of 5 folds. Stage 2 is NOT run here.

In [1]:
import os, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from itgnn.comms_graph import adjacency, build_from_inputs, load_inputs, node_features_v2, pilot_curves_v2, shuffle_edges
from itgnn.data import load_config
from itgnn.evaluation import alert_budget, evaluate_scores, make_folds
from itgnn.weekly_features import drop_warmup
from itgnn.weekly_gnn import choose_epoch, train_scores
from itgnn.weekly_graph import fold_masks

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
cfg = load_config("configs/config.yaml"); SEED = cfg["seed"]; gcfg = cfg["gnn"]; QS = [0.25, 0.5, 1, 2]
LOG = open(os.environ.get("PROBE_LOG", os.devnull), "a"); T0 = time.time()
def tick(s): print(f"[{s}] {time.time() - T0:.0f}s", file=LOG, flush=True)
pd.options.display.float_format = "{:,.3f}".format; pd.options.display.width = 220; pd.options.display.max_columns = 40
torch.set_num_threads(os.cpu_count())
b0 = json.load(open("results/graph_v2_b0.json")); K = b0["chosen_K"]; assert K == 5 and b0["use_email"]
inp = load_inputs(cfg); d = inp["d"]; labels = d["labels"]
g = build_from_inputs(inp, K, K); A = adjacency(g)
ins = pd.read_csv(Path(cfg["raw_dir"]) / "answers" / "insiders.csv")
scen = ins[ins["dataset"].astype(str) == "4.2"].set_index("user")["scenario"]; scen_of = scen.to_dict()
folds = [f for f in make_folds(d["users"], labels, cfg["cv"]["n_splits"], cfg["cv"]["n_repeats"], SEED) if f["repeat"] == 0]
weekly = drop_warmup(inp["full"]); keep = g.rows[g.rows["has_history"]][["user", "week_start"]].sort_values(["user", "week_start"]).reset_index(drop=True)
pd.testing.assert_frame_equal(keep, weekly[["user", "week_start"]].sort_values(["user", "week_start"]).reset_index(drop=True))
print(f"graph: {g.n_user:,} user-week + {g.n_pc:,} PC-week + {g.n_dom:,} domain-week nodes; undirected edges PC {int((g.etype == 0).sum()):,}, domain {int((g.etype == 1).sum()):,}, email {int((g.etype == 2).sum()):,}; scored rows {len(keep):,} (same as the tabular runs)")
tick("graph built")

/home/nothing/insider-threat-gnn/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


graph: 67,238 user-week + 67,375 PC-week + 24,999 domain-week nodes; undirected edges PC 112,408, domain 336,159, email 322,392; scored rows 63,238 (same as the tabular runs)


/home/nothing/insider-threat-gnn/src/itgnn/comms_graph.py:183: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/Context.cpp:916.)
  return torch.sparse_csr_tensor(torch.from_numpy(A.indptr.astype(np.int64)), torch.from_numpy(A.indices.astype(np.int64)),
/home/nothing/insider-threat-gnn/src/itgnn/comms_graph.py:183: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  return torch.sparse_csr_tensor(torch.from_numpy(A.indptr.

## Pilot (same method as step 5): epoch counts for the MLP and for GraphSAGE; GraphSAGE's count is reused for the shuffled control

In [2]:
fold0 = folds[0]; curves, pil_t = {}, {}
for conv, name in [("mlp", "MLP"), ("wsage", "GraphSAGE")]:
    t = time.time(); curves[name] = pilot_curves_v2(g, A, fold0["train"], gcfg, conv, SEED); pil_t[name] = time.time() - t
    print(f"pilot {name}: {pil_t[name] / 60:.1f} min", file=LOG, flush=True)
pc = pd.concat([c.assign(net=n) for n, c in curves.items()]); pc.to_parquet("results/gnn2_pilot_curves.parquet", index=False)
EPOCHS = {n: choose_epoch(c, gcfg["pilot"]["smooth"], gcfg["pilot"]["tolerance"]) for n, c in curves.items()}
print("chosen epochs:", EPOCHS, "| cap =", gcfg["pilot"]["max_epochs"], "| at the cap (possibly under-trained):", {n: e >= gcfg["pilot"]["max_epochs"] for n, e in EPOCHS.items()})
best = {n: float(c.groupby("epoch")["val_ap"].mean().max()) for n, c in curves.items()}; print("best mean inner-validation PR-AUC (descriptive):", {n: round(v, 3) for n, v in best.items()})
json.dump({"epochs": EPOCHS, "pilot_minutes": {n: round(v / 60, 1) for n, v in pil_t.items()}}, open("results/gnn2_epochs.json", "w"))
# per-epoch cost measured in the pilot (includes the validation forward pass, so it is a slight overestimate)
per_ep = {n: pil_t[n] / (gcfg["pilot"]["n_inner_splits"] * gcfg["pilot"]["max_epochs"]) for n in pil_t}
proj1 = 5 * (EPOCHS["MLP"] * per_ep["MLP"] + 2 * EPOCHS["GraphSAGE"] * per_ep["GraphSAGE"]) / 60
elapsed = (time.time() - T0) / 60
print(f"pilot took {sum(pil_t.values()) / 60:.1f} min; projected stage 1 {proj1:.1f} min; pilot + stage 1 = {sum(pil_t.values()) / 60 + proj1:.1f} min (stop rule: > 60)")
assert sum(pil_t.values()) / 60 + proj1 <= 60, "STOP AND ASK: pilot + stage 1 projected above the pre-registered 60 minutes"
tick("pilot done")

chosen epochs: {'MLP': 64, 'GraphSAGE': 75} | cap = 300 | at the cap (possibly under-trained): {'MLP': False, 'GraphSAGE': False}
best mean inner-validation PR-AUC (descriptive): {'MLP': 0.656, 'GraphSAGE': 0.66}
pilot took 16.7 min; projected stage 1 11.0 min; pilot + stage 1 = 27.7 min (stop rule: > 60)


In [3]:
BLUE, ORANGE, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#e4e3df"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
                     "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True, "font.size": 10, "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb"})
fig, ax = plt.subplots(figsize=(8.6, 4.2))
for n, col in [("GraphSAGE", BLUE), ("MLP", ORANGE)]:
    m = curves[n].groupby("epoch")["val_ap"].mean(); ax.plot(m.index, m.values, color=col, lw=1.3, alpha=0.5)
    sm = m.rolling(gcfg["pilot"]["smooth"], min_periods=gcfg["pilot"]["smooth"]).mean(); ax.plot(sm.index, sm.values, color=col, lw=2, label=f"{n} (smoothed; chosen epoch {EPOCHS[n]})")
    ax.axvline(EPOCHS[n], color=col, lw=1, ls="--")
ax.set_xlabel("epoch"); ax.set_ylabel("inner-validation PR-AUC (mean of 3 inner splits)"); ax.set_title("Pilot curves on the repeat-0 fold-0 training users", loc="left", fontsize=11, color=INK)
ax.legend(frameon=False, loc="lower right"); ax.grid(axis="x", visible=False); fig.tight_layout(); fig.savefig("results/figures/10_pilot_curves.png", dpi=130); plt.close(fig)

## Stage 1: five folds x three models (MLP, GraphSAGE, shuffled-edge GraphSAGE), one seed

In [4]:
y = g.rows["label"].to_numpy(); rows, oof_parts, runs = [], [], []
for f in folds:
    fit, ev = fold_masks(g, f["train"], f["test"]); X, _ = node_features_v2(g, fit); loss_idx = np.flatnonzero(fit)
    gs, share = shuffle_edges(g, SEED + f["fold"]); As = adjacency(gs)
    for name, conv, M, ep in [("M0 MLP", "mlp", A, EPOCHS["MLP"]), ("M1 GraphSAGE", "wsage", A, EPOCHS["GraphSAGE"]), ("M2 GraphSAGE shuffled", "wsage", As, EPOCHS["GraphSAGE"])]:
        t = time.time(); s_all, _ = train_scores(X, M, y, loss_idx, ep, conv, gcfg, SEED); dt = time.time() - t
        s = s_all[: g.n_user][ev]; ev_ = evaluate_scores(y[ev], s, k=100)
        rows.append({"model": name, "fold": f["fold"], **ev_}); runs.append({"model": name, "fold": f["fold"], "seconds": dt, "epochs": ep, "shuffle email self-loop share": share if "shuffled" in name else np.nan})
        oof_parts.append(pd.DataFrame({"user": g.rows["user"].to_numpy()[ev], "week_start": g.rows["week_start"].to_numpy()[ev], "repeat": 0, "fold": f["fold"], "model": name, "score": s, "label": y[ev]}))
    print(f"fold {f['fold'] + 1}/5 done, elapsed {(time.time() - T0) / 60:.1f} min", file=LOG, flush=True)
res = pd.DataFrame(rows); oof = pd.concat(oof_parts, ignore_index=True); runs = pd.DataFrame(runs)
oof.to_parquet("results/gnn2_stage1_oof_scores.parquet", index=False)
print(f"stage 1: {runs['seconds'].sum() / 60:.1f} min of training; saved results/gnn2_stage1_oof_scores.parquet {oof.shape}"); display(runs.groupby("model")[["seconds"]].agg(["mean", "sum"]))
tick("stage 1 done")

stage 1: 9.9 min of training; saved results/gnn2_stage1_oof_scores.parquet (189714, 7)


seconds        
                         mean     sum
model                                
M0 MLP                 23.713 118.567
M1 GraphSAGE           46.961 234.805
M2 GraphSAGE shuffled  48.292 241.461

## Results against the pre-registered gate

In [5]:
oof25 = pd.read_parquet("results/comms_25fold_oof_scores.parquet"); cmp_models = ["XGB W2+E+H", "XGB W2+E+H generic", "RF W2+E+H"]
ref = oof25[(oof25["repeat"] == 0) & oof25["model"].isin(cmp_models)][["user", "week_start", "repeat", "fold", "model", "score", "label"]]
for f in folds:  # same folds, same rows
    a = set(map(tuple, oof[(oof["fold"] == f["fold"]) & (oof["model"] == "M1 GraphSAGE")][["user", "week_start"]].to_numpy()))
    b = set(map(tuple, ref[(ref["fold"] == f["fold"]) & (ref["model"] == "XGB W2+E+H")][["user", "week_start"]].to_numpy())); assert a == b
for m in cmp_models:
    r = ref[ref["model"] == m]
    res = pd.concat([res, pd.DataFrame([{"model": m, "fold": fo, **evaluate_scores(gg["label"], gg["score"], 100)} for fo, gg in r.groupby("fold")])], ignore_index=True)
order = ["M0 MLP", "M1 GraphSAGE", "M2 GraphSAGE shuffled"] + cmp_models
S = res.groupby("model")[["pr_auc", "roc_auc", "precision_at_k", "recall_at_k"]].agg(["mean", "std"]).loc[order]
display(pd.DataFrame({m: S[m].apply(lambda r: f"{r['mean']:.3f} ± {r['std']:.3f}", axis=1) for m in ["pr_auc", "roc_auc", "precision_at_k", "recall_at_k"]}))
pv = res.pivot_table(index="fold", columns="model", values="pr_auc")[order]; print("per-fold PR-AUC:"); display(pv)
def pair(a, b): dl = pv[a] - pv[b]; return {"mean diff": dl.mean(), "std": dl.std(), "folds positive": f"{int((dl > 0).sum())}/5", **{f"fold {i}": dl[i] for i in dl.index}}
pairs = [("M1 GraphSAGE", "M0 MLP"), ("M1 GraphSAGE", "M2 GraphSAGE shuffled"), ("M2 GraphSAGE shuffled", "M0 MLP"), ("M1 GraphSAGE", "XGB W2+E+H"), ("M1 GraphSAGE", "XGB W2+E+H generic"), ("M1 GraphSAGE", "RF W2+E+H"), ("M0 MLP", "XGB W2+E+H")]
display(pd.DataFrame({f"{a} - {b}": pair(a, b) for a, b in pairs}).T)
g0, g2 = pv["M1 GraphSAGE"] - pv["M0 MLP"], pv["M1 GraphSAGE"] - pv["M2 GraphSAGE shuffled"]
gate = {"M1 - M0: mean >= +0.05": (g0.mean(), g0.mean() >= 0.05), "M1 - M0: positive in >= 4/5 folds": (int((g0 > 0).sum()), (g0 > 0).sum() >= 4),
        "M1 - M2: mean >= +0.05": (g2.mean(), g2.mean() >= 0.05), "M1 - M2: positive in >= 4/5 folds": (int((g2 > 0).sum()), (g2 > 0).sum() >= 4)}
display(pd.DataFrame(gate, index=["value", "met"]).T)
print("GATE:", "PASSED (stage 2 needs the owner's approval)" if all(v[1] for v in gate.values()) else "NOT PASSED: stage 2 is not run")

,pr_auc,roc_auc,precision_at_k,recall_at_k
model,,,,
M0 MLP,0.708 ± 0.155,0.967 ± 0.047,0.460 ± 0.114,0.717 ± 0.174
M1 GraphSAGE,0.718 ± 0.176,0.973 ± 0.038,0.460 ± 0.102,0.718 ± 0.168
M2 GraphSAGE shuffled,0.717 ± 0.167,0.981 ± 0.023,0.462 ± 0.115,0.721 ± 0.186
XGB W2+E+H,0.879 ± 0.080,0.998 ± 0.001,0.574 ± 0.036,0.897 ± 0.100
XGB W2+E+H generic,0.824 ± 0.112,0.995 ± 0.005,0.546 ± 0.052,0.854 ± 0.125
RF W2+E+H,0.743 ± 0.098,0.997 ± 0.002,0.516 ± 0.078,0.807 ± 0.148


per-fold PR-AUC:


model,M0 MLP,M1 GraphSAGE,M2 GraphSAGE shuffled,XGB W2+E+H,XGB W2+E+H generic,RF W2+E+H
fold,,,,,,
0,0.834,0.838,0.832,0.841,0.788,0.741
1,0.781,0.796,0.774,0.928,0.916,0.833
2,0.830,0.887,0.884,0.979,0.939,0.773
3,0.480,0.475,0.478,0.769,0.658,0.579
4,0.615,0.594,0.620,0.878,0.819,0.787


,mean diff,std,folds positive,fold 0,fold 1,fold 2,fold 3,fold 4
M1 GraphSAGE - M0 MLP,0.010,0.030,3/5,0.003,0.014,0.057,-0.006,-0.021
M1 GraphSAGE - M2 GraphSAGE shuffled,0.000,0.017,3/5,0.006,0.022,0.003,-0.003,-0.025
M2 GraphSAGE shuffled - M0 MLP,0.009,0.025,2/5,-0.003,-0.008,0.054,-0.003,0.004
M1 GraphSAGE - XGB W2+E+H,-0.161,0.126,0/5,-0.004,-0.132,-0.091,-0.294,-0.283
M1 GraphSAGE - XGB W2+E+H generic,-0.106,0.109,1/5,0.050,-0.120,-0.051,-0.184,-0.225
M1 GraphSAGE - RF W2+E+H,-0.025,0.131,2/5,0.097,-0.038,0.114,-0.104,-0.193
M0 MLP - XGB W2+E+H,-0.170,0.112,0/5,-0.007,-0.146,-0.148,-0.289,-0.262


,value,met
M1 - M0: mean >= +0.05,0.010,False
M1 - M0: positive in >= 4/5 folds,3,False
M1 - M2: mean >= +0.05,0.000,False
M1 - M2: positive in >= 4/5 folds,3,False


GATE: NOT PASSED: stage 2 is not run


## Alert budget (repeat 0; caught of the insiders with attack weeks / normal users flagged of 930) and per scenario

In [6]:
allo = pd.concat([oof, ref], ignore_index=True)
bud = alert_budget(allo, labels, scen_of, QS, SEED)
display(bud.set_index(["model", "q%"])[["caught", "normal flagged", "caught s1", "caught s2", "caught s3", "weeks flagged"]].loc[order, :])
print("eligible insiders: 70 (s1 30, s2 30, s3 10); normal users 930")
tick("results")

caught  normal flagged  caught s1  caught s2  caught s3  weeks flagged
model                 q%                                                                           
M0 MLP                0.250      63               4         30         24          9            161
                      0.500      65              16         30         26          9            319
                      1.000      65              46         30         26          9            634
                      2.000      66              82         30         27          9           1266
M1 GraphSAGE          0.250      59               4         30         20          9            161
                      0.500      63              22         30         24          9            319
                      1.000      65              48         30         26          9            634
                      2.000      67              90         30         28          9           1266
M2 GraphSAGE shuffled 0.250      60               4         30         21          9            161
                      0.500      65              24         30         26          9            319
                      1.000      65              46         30         26          9            634
                      2.000      67              79         30         28          9           1266
XGB W2+E+H            0.250      56               0         30         24          2            161
                      0.500      62               7         30         28          4            319
                      1.000      68              27         30         30          8            634
                      2.000      69              51         30         30          9           1266
XGB W2+E+H generic    0.250      50               1         20         26          4            161
                      0.500      59              11         27         28          4            319
                      1.000      66              23         30         29          7            634
                      2.000      67              63         30         29          8           1266
RF W2+E+H             0.250      41               2         19         19          3            161
                      0.500      58               6         28         24          6            319
                      1.000      66              20         30         29          7            634
                      2.000      70              67         30         30         10           1266

eligible insiders: 70 (s1 30, s2 30, s3 10); normal users 930


In [7]:
fig, ax = plt.subplots(figsize=(8.6, 4.4)); labs = order[::-1]
cols = {"M0 MLP": ORANGE, "M1 GraphSAGE": BLUE, "M2 GraphSAGE shuffled": "#6250d6", "XGB W2+E+H": "#1baf7a", "XGB W2+E+H generic": "#eda100", "RF W2+E+H": "#e87ba4"}
for i, m in enumerate(labs):
    v = pv[m]; ax.scatter(v.values, [i] * 5, s=38, color=cols[m], alpha=0.9, edgecolor="#fcfcfb", linewidth=1.2, zorder=3); ax.plot([v.mean()] * 2, [i - 0.25, i + 0.25], color=cols[m], lw=2.4, zorder=4)
ax.set_yticks(range(len(labs))); ax.set_yticklabels(labs); ax.set_xlabel("PR-AUC per test fold (dots) and mean (bar), repeat 0"); ax.grid(axis="y", visible=False)
ax.set_title("Stage 1: graph vs no-edge MLP vs shuffled edges vs tabular", loc="left", fontsize=11, color=INK); fig.tight_layout(); fig.savefig("results/figures/10_stage1_pr_auc.png", dpi=130); plt.close(fig)